# Controller sensitivity

Vary one controller setting at a time while keeping the aggressive policy,
kappa=2 and five policy seeds fixed across the four test regimes. The analysis
uses the saved episode records and does not retrain the policy.

Section 6.6 reports the baseline and the half-life, IV-scale and ACI-step
variations: seven settings across four regimes. This notebook also evaluates
epsilon and warm-up variations, giving 11 settings in total.


## 1. Locate the experiment directory

Start Jupyter from the repository root.


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


matplotlib_cache = Path('/tmp/v13-sensitivity-matplotlib')
matplotlib_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(matplotlib_cache))

REGIMES = (
    'crash_test2018',
    'crash_test2020',
    'mix_test2022',
    'mix_test2024',
)

def _is_sensitivity_root(candidate):
    candidate = Path(candidate).expanduser()
    script_ok = (candidate / 'new-reward' / 'v13_sensitivity_replay.py').is_file()
    reference_ok = (
        candidate / 'results' / 'warm0_batch_v13'
        / 'table5_aggressive_validation_budget.csv'
    ).is_file()
    ledgers_ok = all(
        (
            candidate / 'new-reward' / 'checkpoints'
            / f'conformal_controls_v12r_{regime}'
            / f'ledger_arm00_{regime}.npz'
        ).is_file()
        for regime in REGIMES
    )
    return script_ok and reference_ok and ledgers_ok

def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'ctc_runtime.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the conformal-trust-control repository.')

KIT_ROOT = _find_repository_root()
SCRIPT = KIT_ROOT / 'new-reward' / 'v13_sensitivity_replay.py'
OUTPUT = KIT_ROOT / 'results' / 'v13_sensitivity'
print('Repository:', KIT_ROOT)
print('sensitivity script:', SCRIPT)
print('output directory:', OUTPUT)

## 2. Parameter settings

The default setting is shared across the one-factor comparisons.


In [ ]:
import pandas as pd
from IPython.display import display

design = pd.DataFrame([
    ('Target breach probability epsilon', '0.05, [0.10], 0.20', 'Risk appetite; validation budget is recalibrated'),
    ('Warm-up episodes', '10, [15], 30', 'Cold-start conservatism before leaving the exact fallback'),
    ('Recency half-life (days)', '15, [30], 60', 'Short versus persistent memory of old regimes'),
    ('IV similarity scale tau_IV', '0.02, [0.04], 0.08', 'Locality of evidence transfer across volatility states'),
    ('Batched-ACI step gamma', '0.01, [0.02], 0.04', 'Risk-feedback responsiveness versus stability'),
], columns=['factor', 'values (default in brackets)', 'design meaning'])
display(design)

## 3. Run the sensitivity analysis

Evaluate 11 settings across four regimes and five policy seeds. Uncertainty estimates use 3,000 moving-block bootstrap replicates over blocks of 10 start-date batches.


In [ ]:
import runpy
import time

started = time.perf_counter()
sensitivity = runpy.run_path(str(SCRIPT))
sensitivity['run'](
    root=KIT_ROOT,
    output=OUTPUT,
    kappa=2,
    bootstrap_reps=3000,
    bootstrap_block_batches=10,
    make_plot=True,
)
print(f'Completed in {time.perf_counter() - started:.1f} seconds')

## 4. Check dimensions and default-setting agreement


In [ ]:
import hashlib
import json

seed_level = pd.read_csv(OUTPUT / 'sensitivity_seed_level.csv')
panel = pd.read_csv(OUTPUT / 'sensitivity_panel.csv')
summary = pd.read_csv(OUTPUT / 'sensitivity_summary.csv')
manifest = json.loads((OUTPUT / 'manifest.json').read_text(encoding='utf-8'))

assert len(seed_level) == 11 * 4 * 5, len(seed_level)
assert len(panel) == 11 * 4, len(panel)
assert len(summary) == 11, len(summary)
assert not seed_level.duplicated(['config_id', 'regime', 'seed']).any()
assert not panel.duplicated(['config_id', 'regime']).any()
assert manifest['default_regression']['status'] == 'PASS'
assert manifest['default_regression']['numeric_checks'] == 40
assert manifest['training_performed'] is False
actual_script_hash = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
assert actual_script_hash == manifest['source_sha256']['new-reward/v13_sensitivity_replay.py']

audit = {
    'status': 'PASS',
    'unique configurations': int(summary['config_id'].nunique()),
    'seed-level rows': len(seed_level),
    'regime-panel rows': len(panel),
    'default regression checks': manifest['default_regression']['numeric_checks'],
    'training performed': manifest['training_performed'],
}
display(pd.Series(audit, name='audit'))

## 5. Sensitivity summary

The point-target gap is the observed breach rate minus epsilon. The upper-target gap uses the one-sided 95% bootstrap bound. Both are reported in percentage points.


In [ ]:
chapter6_table = summary[[
    'config_id',
    'minimum_regime_mean_bp',
    'maximum_regime_breach',
    'maximum_breach_target_gap',
    'maximum_date_block_upper_target_gap',
    'all_regimes_point_feasible',
]].copy()
chapter6_table['maximum_regime_breach_pct'] = 100 * chapter6_table.pop('maximum_regime_breach')
chapter6_table['point_target_gap_pp'] = 100 * chapter6_table.pop('maximum_breach_target_gap')
chapter6_table['upper_95_target_gap_pp'] = 100 * chapter6_table.pop('maximum_date_block_upper_target_gap')
chapter6_table = chapter6_table[[
    'config_id',
    'minimum_regime_mean_bp',
    'maximum_regime_breach_pct',
    'point_target_gap_pp',
    'upper_95_target_gap_pp',
    'all_regimes_point_feasible',
]]
display(chapter6_table.round(4))

## 6. Plot sensitivity results


In [ ]:
from IPython.display import Image

display(Image(filename=str(OUTPUT / 'sensitivity_worst_regime.png')))
print('PDF figure:', OUTPUT / 'sensitivity_worst_regime.pdf')
print('CSV summary:', OUTPUT / 'sensitivity_summary.csv')
print('Audit manifest:', OUTPUT / 'manifest.json')

## Interpretation

These comparisons assess sensitivity around the selected controller configuration. They are not used to choose a new setting from held-out test results. Point breach rates and their one-sided confidence bounds are reported separately.
